# **Phase 7 — Full RAG Pipeline & Evaluation**

## **0. Install Dependencies**

In [ ]:
# %pip uninstall -y transformers sentence-transformers FlagEmbedding unsloth unsloth_zoo bitsandbytes
# %pip uninstall -y faiss-cpu bm25s bert-score ragas langchain-openai accelerate bitsandbytes pandas pyarrow numpy tqdm matplotlib seaborn
# %pip cache purge

In [ ]:
# !rm -rf root/.cache

In [ ]:
# %pip install -q unsloth transformers sentence-transformers FlagEmbedding
# %pip install -q faiss-cpu bm25s bert-score ragas langchain-openai accelerate bitsandbytes pandas pyarrow numpy tqdm matplotlib seaborn

In [ ]:
# !pip show transformers

## **1. Setup**

In [1]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES']    = '0'
os.environ['TOKENIZERS_PARALLELISM']  = 'false'

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import unsloth

import gc
import re
import json
import random
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib   import Path
from tqdm.auto import tqdm

import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')
print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [2]:
# ===== Paths =====
DATA_DIR      = Path('workspace/data/cleaned')
PROCESSED_DIR = Path('workspace/data/processed')
OUTPUT_DIR    = Path('workspace/results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===== HuggingFace model IDs =====
BIENCODER_ID  = 'YuITC/vietnamese-embedding-vn-legal'
RERANKER_ID   = 'YuITC/bge-reranker-v2-m3-vn-legal'
GENERATOR_ID  = 'YuITC/gemma4-e4b-it-vn-legal-16bit'

# ===== Retrieval hyperparams (best from Phase 5) =====
RETRIEVAL_K   = 30         # initial candidate pool
FINAL_K       = 3          # after reranking
RRF_K         = 10         # RRF damping constant (tuned in Phase 5)
RRF_WEIGHTS   = [0.2, 1.8] # [BM25, Dense] weights (tuned in Phase 5)

# ===== Evaluation sample size =====
EVAL_N        = 1000
RAGAS_N       = 1000

## **2. Load Data**

In [3]:
corpus    = pd.read_parquet(DATA_DIR / 'corpus.parquet')
val_split = pd.read_parquet(DATA_DIR / 'val_split.parquet')

corpus = corpus.reset_index(drop=True)

corpus_texts = corpus['text'].tolist()
corpus_cids  = corpus['cid'].tolist()

cid2idx  = {c: i for i, c in enumerate(corpus_cids)}
idx2cid  = {i: c for c, i in cid2idx.items()}
cid2text = dict(zip(corpus_cids, corpus_texts))

# Normalise cid lists in val_split
val_split['cid'] = val_split['cid'].apply(
    lambda x: [int(i) for i in (x.tolist() if isinstance(x, np.ndarray) else x)]
)

# Sample evaluation subset
val_sample    = val_split.sample(n=EVAL_N, random_state=SEED).reset_index(drop=True)
val_questions = val_sample['question'].tolist()
val_relevant  = val_sample['cid'].tolist()     # list[list[int]]
val_contexts  = val_sample['context'].tolist() # list[list[str]] - ground-truth texts

print(f'Corpus    : {len(corpus):,} documents')
print(f'Val full  : {len(val_split):,} queries')
print(f'Val sample: {len(val_sample):,} queries')

In [ ]:
# tmp = val_sample.copy()
# tmp['question_len'] = tmp['question'].apply(len)
# tmp['context_len'] = tmp['context'].apply(
#     lambda c: sum(len(text) for text in c)
# )
# tmp.describe()

## **3. Load Retrieval Artifacts**

In [ ]:
# import faiss
# import bm25s
# from sentence_transformers import SentenceTransformer
# from FlagEmbedding         import FlagReranker

In [ ]:
# # ===== BM25 =====
# corpus_tokens = pd.read_parquet(PROCESSED_DIR / 'bm25_corpus_tokens.parquet')['tokens'].tolist()
# val_tokens    = pd.read_parquet(PROCESSED_DIR / 'bm25_val_tokens.parquet')['tokens'].tolist()

# # Pick the val tokens corresponding to our sample
# sample_indices  = val_sample.index.tolist()
# val_tokens_smpl = [val_tokens[i] for i in sample_indices]

# corpus_tokens   = [list(map(str, x)) for x in corpus_tokens]
# val_tokens_smpl = [list(map(str, x)) for x in val_tokens_smpl]

# bm25 = bm25s.BM25(k1=1.5, b=0.75)
# bm25.index(corpus_tokens)

In [ ]:
# # ===== FAISS (fine-tuned bi-encoder) =====
# faiss_index     = faiss.read_index(str(PROCESSED_DIR / 'ft_faiss_index.bin'))

# val_embeds      = np.load(PROCESSED_DIR / 'ft_val_embeddings.npy', mmap_mode='r')
# val_embeds_smpl = val_embeds[sample_indices]

In [ ]:
# # ===== Fine-tuned Bi-Encoder (for query re-encoding at inference time) =====
# print(f'Loading bi-encoder: {BIENCODER_ID} ...')

# bi_encoder = SentenceTransformer(
#     BIENCODER_ID,
#     model_kwargs={
#         'torch_dtype'        : torch.bfloat16,
#         'attn_implementation': 'eager',
#     }
# )
# bi_encoder.max_seq_length = 512
# bi_encoder.eval()

In [ ]:
# # ===== FlagReranker (fine-tuned cross-encoder) =====
# print(f'Loading reranker: {RERANKER_ID} ...')

# reranker = FlagReranker(
#     RERANKER_ID, 
#     use_fp16=True
# )

## **4. Full Retrieval Pipeline**

In [ ]:
# def rrf_fusion(ranked_lists: list, k: int = RRF_K, weights: list = RRF_WEIGHTS) -> list:
#     """Reciprocal Rank Fusion across multiple ranked lists."""
#     scores = {}
#     for ranked, w in zip(ranked_lists, weights):
#         for rank, cid in enumerate(ranked):
#             scores[cid] = scores.get(cid, 0.0) + w / (rank + 1 + k)
#     return sorted(scores, key=lambda c: scores[c], reverse=True)


# def rerank_batch(
#     run_cids  : list,
#     questions : list,
#     cid2text  : dict,
#     reranker,
#     top_k     : int = FINAL_K,
#     batch_size: int = 128,
# ) -> list:
#     """Batch reranking — returns top_k cids per query."""
#     all_pairs, query_doc_counts = [], []

#     for q, cids in zip(questions, run_cids):
#         docs = [cid2text.get(c, '') for c in cids]
#         all_pairs.extend([[q, d] for d in docs])
#         query_doc_counts.append(len(cids))

#     print(f'  Reranking {len(all_pairs):,} query-doc pairs ...')

#     gc.collect()
#     torch.cuda.empty_cache()

#     all_scores = reranker.compute_score(all_pairs, batch_size=batch_size, max_length=512)

#     del all_pairs
#     gc.collect()

#     reranked_runs = []
#     start_idx = 0
#     for cids, num_docs in zip(run_cids, query_doc_counts):
#         scores       = all_scores[start_idx : start_idx + num_docs]
#         sorted_idx   = np.argsort(scores)[::-1]
#         reranked_runs.append([cids[j] for j in sorted_idx][:top_k])
#         start_idx   += num_docs

#     return reranked_runs

In [ ]:
# bm25_results, _ = bm25.retrieve(val_tokens_smpl, corpus_cids, RETRIEVAL_K, n_threads=os.cpu_count())
# bm25_run        = [[int(x) for x in row] for row in bm25_results.tolist()]

In [ ]:
# _, I_dense = faiss_index.search(val_embeds_smpl.astype(np.float32), RETRIEVAL_K)
# dense_run  = [[idx2cid[i] for i in row if i >= 0] for row in I_dense]

In [ ]:
# fusion_run = [
#     rrf_fusion([bm25_run[i], dense_run[i]], k=RRF_K, weights=RRF_WEIGHTS)
#     for i in range(EVAL_N)
# ]

In [ ]:
# reranked_run = rerank_batch(fusion_run, val_questions, cid2text, reranker, top_k=FINAL_K)

In [ ]:
# import pickle

# Path('workspace/data/tmp').mkdir(parents=True, exist_ok=True)

# with open('workspace/data/tmp/reranked_run.pkl', 'wb') as f:
#     pickle.dump(reranked_run, f)

## **5. Answer Generation (Fine-tuned Gemma4)**

In [4]:
import pickle

with open('workspace/data/tmp/reranked_run.pkl', 'rb') as f:
    reranked_run = pickle.load(f)

In [5]:
from transformers import AutoProcessor, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from unsloth import FastLanguageModel

SYSTEM_PROMPT = """Bạn là một chuyên gia tư vấn pháp luật Việt Nam giàu kinh nghiệm. 
Nhiệm vụ của bạn là trả lời câu hỏi pháp lý dựa vào CHÍNH XÁC và DUY NHẤT vào các đoạn văn bản được cung cấp.

Quy tắc bắt buộc:
1. Chỉ sử dụng thông tin từ context đã cho — không dùng kiến thức ngoài.
2. Trả lời bằng tiếng Việt tự nhiên, rõ ràng, đúng ngữ pháp.
3. Cuối câu trả lời, liệt kê citations theo format:
   [Nguồn N: <trích dẫn ngắn gọn từ văn bản N>]
4. Nếu context không đủ để trả lời, nói rõ: "Thông tin trong văn bản chưa đủ để trả lời câu hỏi này."
5. KHÔNG bịa đặt thông tin, KHÔNG suy diễn ngoài văn bản.
6. KHÔNG xưng hô hay lòng vòng, đi thẳng vào câu trả lời.
7. KHÔNG sử dụng các cụm từ thể hiện bản thân là AI."""

In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit           = True,
    bnb_4bit_quant_type    = 'nf4',
    bnb_4bit_compute_dtype = torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(GENERATOR_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

generator = AutoModelForCausalLM.from_pretrained(
    GENERATOR_ID,
    quantization_config = bnb_config,
    device_map          = {'': 0},
    torch_dtype         = torch.bfloat16,
    attn_implementation = 'sdpa',
).eval()

In [11]:
def build_user_prompt(question: str, passages: list) -> str:
    parts = [f"[Văn bản {i+1}]\n{p}" for i, p in enumerate(passages)]
    ctx   = '\n\n'.join(parts)
    return f"Câu hỏi: {question}\n\nVăn bản pháp luật liên quan:\n{ctx}"

def truncate_passage(text, tokenizer, max_tokens=256):
    tokens = tokenizer.encode(text, truncation=True, max_length=max_tokens)
    return tokenizer.decode(tokens, skip_special_tokens=True)

In [16]:
@torch.no_grad()
def generate_answers_batch(
    questions     : list,
    run_cids      : list,
    cid2text      : dict,
    model,
    tokenizer,
    batch_size    : int = 8,
    max_new_tokens: int = 256,
) -> list:
    """
    Generate answers for a list of (question, retrieved_cids) pairs.
    Returns a list of answer strings.
    """
    answers = []

    for start in tqdm(range(0, len(questions), batch_size), desc='Generating'):
        batch_q    = questions[start : start + batch_size]
        batch_cids = run_cids[start  : start + batch_size]

        texts = []
        for q, cids in zip(batch_q, batch_cids):
            passages = [cid2text.get(c, '') for c in cids[:FINAL_K]]
            # passages = [
            #     truncate_passage(cid2text.get(c, ''), tokenizer) 
            #     for c in cids[:FINAL_K]
            # ]
            messages = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user'  , 'content': build_user_prompt(q, passages)},
            ]
            text = tokenizer.apply_chat_template(
                messages,
                tokenize              = False,
                add_generation_prompt = True,
                enable_thinking       = False
            )
            texts.append(text)

        inputs = tokenizer(
            texts,
            return_tensors = 'pt',
            padding        = True,
            truncation     = True,
            max_length     = 2048,
        ).to('cuda')

        input_len = inputs['input_ids'].shape[-1]

        outputs = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = 1.0,
            top_p          = 0.95,
            top_k          = 64,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )

        generated = outputs[:, input_len:]
        decoded   = tokenizer.batch_decode(generated, skip_special_tokens=True)
        answers.extend(decoded)

        # del inputs, outputs, generated
        # gc.collect()
        # torch.cuda.empty_cache()

    return answers

In [17]:
# Generate answers using the FULL pipeline (reranked top-FINAL_K contexts)
generated_answers = generate_answers_batch(
    questions      = val_questions,
    run_cids       = reranked_run,
    cid2text       = cid2text,
    model          = generator,
    tokenizer      = tokenizer,
    batch_size     = 4,
    max_new_tokens = 256,
)

print(f'\nGenerated {len(generated_answers)} answers.')
print('\n--- Sample answer ---')
print(f'Q: {val_questions[0]}')
print(f'A: {generated_answers[0][:500]}...')

In [19]:
# Persist results for downstream evaluation
rag_results = pd.DataFrame({
    'qid'             : val_sample['qid'].tolist(),
    'question'        : val_questions,
    'relevant_cids'   : val_relevant,
    'retrieved_cids'  : reranked_run,
    'retrieved_texts' : [[cid2text.get(c, '') for c in cids] for cids in reranked_run],
    'ground_truth_ctx': val_contexts,
    'answer'          : generated_answers,
})

## **6. BERTScore Evaluation**

In [ ]:
from bert_score import score as bert_score_fn

# Reference = ground-truth context passages joined
# (proxy for ideal answer since no gold answers exist)
references = [
    ' '.join(ctx_list) if isinstance(ctx_list, list) else str(ctx_list)
    for ctx_list in val_contexts
]

print('Computing BERTScore (this may take a few minutes) ...')

P, R, F1 = bert_score_fn(
    cands      = generated_answers,
    refs       = references,
    lang       = 'vi', # Vietnamese
    model_type = 'bert-base-multilingual-cased',
    verbose    = True,
    batch_size = 32,
    device     = 'cuda',
)

In [ ]:
bertscore_results = {
    'BERTScore Precision': float(P.mean()),
    'BERTScore Recall'   : float(R.mean()),
    'BERTScore F1'       : float(F1.mean()),
}

print('\n===== BERTScore =====')
for k, v in bertscore_results.items():
    print(f'  {k}: {v:.4f}')

rag_results['bertscore_P']  = P.tolist()
rag_results['bertscore_R']  = R.tolist()
rag_results['bertscore_F1'] = F1.tolist()

## **7. Citation Accuracy**

In [ ]:
def extract_citations(answer: str) -> list:
    """
    Parse citation indices from the answer.
    Expected format: [Nguồn N: ...]
    Returns a list of 1-based citation indices (int).
    """
    # Match [Nguồn N: ...] where N is an integer
    pattern = r'\[Nguồn\s+(\d+)\s*:'
    matches = re.findall(pattern, answer, re.IGNORECASE)
    return [int(m) for m in matches]


def citation_accuracy(
    answer        : str,
    retrieved_cids: list, # ordered list of retrieved cids (position = citation index)
    relevant_cids : list, # ground-truth relevant cids
) -> dict:
    """
    Citation Precision: cited docs that are actually relevant / total cited docs
    Citation Recall   : relevant docs that were cited   / total relevant docs
    Citation F1       : harmonic mean
    has_citation      : 1 if at least one citation found, else 0
    """
    cited_indices = extract_citations(answer) # 1-based
    relevant_set  = set(relevant_cids)

    # Convert 1-based citation index → cid
    cited_cids = [
        retrieved_cids[idx - 1]
        for idx in cited_indices
        if 1 <= idx <= len(retrieved_cids)
    ]
    cited_set = set(cited_cids)

    tp   = len(cited_set & relevant_set)
    prec = tp / len(cited_set)    if cited_set    else 0.0
    rec  = tp / len(relevant_set) if relevant_set else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

    return {
        'citation_precision': prec,
        'citation_recall'   : rec,
        'citation_f1'       : f1,
        'has_citation'      : int(len(cited_indices) > 0),
        'n_citations'       : len(cited_indices),
    }

In [ ]:
# Run citation evaluation
citation_scores = [
    citation_accuracy(ans, r_cids, gt_cids)
    for ans, r_cids, gt_cids in zip(generated_answers, reranked_run, val_relevant)
]

df_citation = pd.DataFrame(citation_scores)

citation_results = df_citation.mean().to_dict()

print('\n===== Citation Accuracy =====')
for k, v in citation_results.items():
    print(f'  {k}: {v:.4f}')

rag_results = rag_results.join(df_citation)

## **8. RAGAS Evaluation**

> **Metrics used**:
> - `faithfulness`         — Does the answer stick to the context? *(LLM-based)*
> - `answer_relevancy`     — Is the answer relevant to the question? *(LLM-based)*
> - `context_precision`    — Are relevant docs ranked high in the context? *(LLM-based)*
> - `context_recall`       — Is all ground-truth information covered by the retrieved context? *(LLM-based)*

In [ ]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

OPENAI_API_KEY = ''

ragas_llm        = ChatOpenAI(model='gpt-4o-mini', api_key=OPENAI_API_KEY)
ragas_embeddings = OpenAIEmbeddings(model='text-embedding-3-small', api_key=OPENAI_API_KEY)

In [ ]:
from ragas import evaluate as ragas_evaluate
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    LLMContextPrecisionWithoutReference,
    ContextRecall,
)

# ===== Build RAGAS dataset (smaller subset to control cost) =====
ragas_idx = random.sample(range(EVAL_N), RAGAS_N)

ragas_samples = []
for i in ragas_idx:
    # ground_truth = relevant context passages joined (proxy for reference answer)
    gt_texts = val_contexts[i]
    if not isinstance(gt_texts, list):
        gt_texts = [str(gt_texts)]
    reference = ' '.join(gt_texts)

    ragas_samples.append(
        SingleTurnSample(
            user_input          = val_questions[i],
            response            = generated_answers[i],
            retrieved_contexts  = [cid2text.get(c, '') for c in reranked_run[i]],
            reference           = reference,
        )
    )

ragas_dataset = EvaluationDataset(samples=ragas_samples)
print(f'RAGAS dataset: {len(ragas_dataset)} samples')

In [ ]:
# ── Instantiate metrics with our LLM ──────────────────────────────────────────
faithfulness_metric     = Faithfulness(llm=ragas_llm)
answer_relevancy_metric = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
ctx_precision_metric    = LLMContextPrecisionWithoutReference(llm=ragas_llm)
ctx_recall_metric       = ContextRecall(llm=ragas_llm)

print('Running RAGAS evaluation ...')

ragas_result = ragas_evaluate(
    dataset = ragas_dataset,
    metrics = [
        faithfulness_metric,
        answer_relevancy_metric,
        ctx_precision_metric,
        ctx_recall_metric,
    ],
    raise_exceptions = False,
)

print('\n===== RAGAS Results =====')
ragas_df = ragas_result.to_pandas()

display(ragas_df[['faithfulness', 'answer_relevancy', 'llm_context_precision_without_reference', 'context_recall']].describe().T)

In [ ]:
ragas_summary = {
    'faithfulness'     : float(ragas_df['faithfulness'].mean()),
    'answer_relevancy' : float(ragas_df['answer_relevancy'].mean()),
    'context_precision': float(ragas_df['llm_context_precision_without_reference'].mean()),
    'context_recall'   : float(ragas_df['context_recall'].mean()),
}

print('\n===== RAGAS Summary (mean) =====')
for k, v in ragas_summary.items():
    print(f'  {k}: {v:.4f}')

ragas_df.to_csv(OUTPUT_DIR / 'phase7_ragas_detailed.csv', index=False)

## **9. Aggregate Results & Final Summary**

In [ ]:
# ── Best retrieval system metrics (full pipeline) ─────────────────────────────
summary = {
    # BERTScore
    '--- BERTScore ---'       : '',
    'BERTScore Precision'     : bertscore_results['BERTScore Precision'],
    'BERTScore Recall'        : bertscore_results['BERTScore Recall'],
    'BERTScore F1'            : bertscore_results['BERTScore F1'],
    # Citation
    '--- Citation Accuracy ---': '',
    'Citation Precision'      : citation_results['citation_precision'],
    'Citation Recall'         : citation_results['citation_recall'],
    'Citation F1'             : citation_results['citation_f1'],
    'Has Citation (%)'        : citation_results['has_citation'] * 100,
    # RAGAS
    '--- RAGAS ---'           : '',
    'Faithfulness'            : ragas_summary['faithfulness'],
    'Answer Relevancy'        : ragas_summary['answer_relevancy'],
    'Context Precision'       : ragas_summary['context_precision'],
    'Context Recall'          : ragas_summary['context_recall'],
}

df_summary = pd.DataFrame(
    [(k, v) for k, v in summary.items()],
    columns=['Metric', 'Score']
)

print('\n' + '='*55)
print('         PHASE 7 — FULL RAG PIPELINE SUMMARY')
print('='*55)
print(f'  Bi-Encoder : {BIENCODER_ID}')
print(f'  Reranker   : {RERANKER_ID}')
print(f'  Generator  : {GENERATOR_ID}')
print(f'  Eval set   : {EVAL_N} queries | RAGAS subset: {RAGAS_N}')
print('='*55)

for _, row in df_summary.iterrows():
    if isinstance(row['Score'], str):
        print(f'\n  {row["Metric"]}')
    else:
        print(f'    {row["Metric"]:<30} {row["Score"]:.4f}')

df_summary.to_csv(OUTPUT_DIR / 'phase7_summary.csv', index=False)
print(f'\nSummary saved → {OUTPUT_DIR / "phase7_summary.csv"}')

In [ ]:
# ── Final dashboard chart ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Phase 7 — Full RAG Pipeline Evaluation', fontsize=14, fontweight='bold')

# Panel 1: Retrieval ablation @ 5
ax = axes[0]
systems  = list(retrieval_results.keys())
r5_vals  = [retrieval_results[s]['recall@5'] for s in systems]
n5_vals  = [retrieval_results[s]['ndcg@5']   for s in systems]
m5_vals  = [retrieval_results[s]['mrr@5']    for s in systems]
x = np.arange(len(systems))
ax.bar(x - 0.25, r5_vals, 0.25, label='Recall@5',  color='steelblue')
ax.bar(x,        n5_vals, 0.25, label='NDCG@5',    color='coral')
ax.bar(x + 0.25, m5_vals, 0.25, label='MRR@5',     color='seagreen')
ax.set_xticks(x)
ax.set_xticklabels([s.replace(' + ', '\n+') for s in systems], fontsize=7)
ax.set_ylim(0, 1)
ax.set_title('Retrieval Ablation @5')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Panel 2: BERTScore & Citation
ax = axes[1]
gen_metrics   = {
    'BERTScore\nF1'         : bertscore_results['BERTScore F1'],
    'Citation\nPrecision'   : citation_results['citation_precision'],
    'Citation\nRecall'      : citation_results['citation_recall'],
    'Citation\nF1'          : citation_results['citation_f1'],
}
bars = ax.bar(gen_metrics.keys(), gen_metrics.values(),
              color=['royalblue', 'tomato', 'gold', 'mediumseagreen'])
ax.set_ylim(0, 1)
ax.set_title('Generation Metrics')
ax.grid(axis='y', alpha=0.3)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

# Panel 3: RAGAS
ax = axes[2]
ragas_metrics = {
    'Faithfulness'      : ragas_summary['faithfulness'],
    'Answer\nRelevancy' : ragas_summary['answer_relevancy'],
    'Context\nPrecision': ragas_summary['context_precision'],
    'Context\nRecall'   : ragas_summary['context_recall'],
}
bars = ax.bar(ragas_metrics.keys(), ragas_metrics.values(),
              color=['darkorchid', 'deepskyblue', 'orangered', 'limegreen'])
ax.set_ylim(0, 1)
ax.set_title('RAGAS Metrics')
ax.grid(axis='y', alpha=0.3)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'phase7_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Dashboard saved → {OUTPUT_DIR / "phase7_dashboard.png"}')

In [ ]:
# ── Save full per-sample results ───────────────────────────────────────────────
rag_results.to_parquet(OUTPUT_DIR / 'phase7_rag_results_final.parquet', index=False)
print(f'Full per-sample results saved → {OUTPUT_DIR / "phase7_rag_results_final.parquet"}')

print('\n✅ Phase 7 complete.')

## **10. Error Analysis**

In [ ]:
# ── Qualitative inspection: worst-performing samples ─────────────────────────
rag_results['ndcg5'] = [
    ndcg_at_k(r, g, 5)
    for r, g in zip(reranked_run, val_relevant)
]

worst = rag_results.nsmallest(5, 'ndcg5')[[
    'question', 'relevant_cids', 'retrieved_cids',
    'ndcg5', 'bertscore_F1', 'citation_f1'
]]

print('=== Worst 5 samples (by NDCG@5) ===')
display(worst)

In [ ]:
# ── Distribution plots ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Score Distributions — Per Sample', fontsize=12)

for ax, col, title, color in zip(
    axes,
    ['bertscore_F1', 'citation_f1', 'ndcg5'],
    ['BERTScore F1', 'Citation F1', 'NDCG@5'],
    ['steelblue', 'coral', 'seagreen'],
):
    if col in rag_results.columns:
        ax.hist(rag_results[col].dropna(), bins=20, color=color, edgecolor='white', alpha=0.85)
        ax.axvline(rag_results[col].mean(), color='black', linestyle='--', linewidth=1.5,
                   label=f'Mean={rag_results[col].mean():.3f}')
        ax.set_title(title)
        ax.set_xlabel('Score')
        ax.set_ylabel('Count')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'phase7_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()